# Project Assignment: Short Video Recommender System (KuaiRec)
Dataset Source: [Kuairec](https://kuairec.com/)

Arxiv Paper: [KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems](https://arxiv.org/pdf/2202.10842)

## Two-Tower Model

Two-Tower is an embedding model used mostly for retrieval tasks, such as search or recommendation.

Two towers refer to the two separate neural networks that are used to encode the user and item features. Each tower is trained independently, and the outputs of the two towers are combined to make predictions.

It is meant to be efficient for large data, and scalable to new users and items.

## Dataset import

The server is down, please download from the Google Drive in the given link.

In [ ]:
!wget https://nas.chongminggao.top:4430/datasets/KuaiRec.zip --no-check-certificate
!unzip KuaiRec.zip

## Imports

In [ ]:
# Misc
import numpy as np
import pandas as pd
from utils import get_data_path, matrix_cleanup 

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA

# Model training
import tensorflow as tf
import keras
from keras.api.layers import Concatenate, Dense, Dot, Dropout, Input
from keras.api.models import Model, Sequential
from keras.api.optimizers import Adam

# Plot metrics
import plotly.express as px
import plotly.graph_objects as go


# Deactivate XLA compilation
tf.config.optimizer.set_jit(False)

# I get my dataset from a Kaggle input
DATA_PATH = get_data_path()
TRAINED_MODEL = "trained/two_tower_model.keras"

DATA_PATH

# Step 1: Load the dataset

## Small matrix

This table has a density of 99.6%. This means that 99.6% of the entries in the matrix are non-zero, indicating that most users have interacted with most items.

In [ ]:
small_matrix = pd.read_csv(f"{DATA_PATH}/small_matrix.csv")

small_matrix = matrix_cleanup(small_matrix)


## Big matrix

This table has a density of 16.3%. We will use this matrix for our training and testing.

It contains more interactions with the same users/items of the small matrix. We do not need to substract the small matrix.

In [ ]:
big_matrix = pd.read_csv(f"{DATA_PATH}/big_matrix.csv")

big_matrix = matrix_cleanup(big_matrix)


## Misc

In [ ]:
print(f"Proportion of small_matrix relative to big_matrix: {small_matrix.shape[0] * 100 / big_matrix.shape[0]:.2f}%")

## Item category encoding

We have the caracteristics of the videos (author_id, video_type...) but this part requires less preprocessing.

For Content-based filtering, we need to use features of the videos (list of tags). We will use a simple one-hot encoding.

In [ ]:
# No missing values for this data
item_categories = pd.read_csv(f"{DATA_PATH}/item_categories.csv")

## Item daily features

This dataset is also interesting for content-based filtering.

Mostly composed of textual data, we will use a TF-IDF vectorizer to encode the features of the videos.

In [ ]:
item_daily_features = pd.read_csv(f"{DATA_PATH}/item_daily_features.csv", lineterminator='\n')
item_daily_features.fillna(-1, inplace=True)

## User features

In [ ]:
user_features = pd.read_csv(f"{DATA_PATH}/user_features.csv", lineterminator='\n')
user_features.fillna(-1, inplace=True)

## Caption Category (Not used for now)

In [ ]:
caption_category = pd.read_csv(f"{DATA_PATH}/kuairec_caption_category.csv", lineterminator='\n')

# Step 2: Feature Engineering

- Create meaningful features from interaction and metadata (e.g., content tags, user activity history)
- Build user-item interaction matrix
- Optionally extract time-based or popularity-based features

## Item category encoding

We have the caracteristics of the videos (author_id, video_type...) but this part requires less preprocessing.

For Content-based filtering, we need to use features of the videos (list of tags). No need for TF-IDF, we will use a simple one-hot encoding.

- **XXX_features** : The Dataframe with all features, used initially
- **XXX_features_map** : The Dataframe with the mapping of the features, filtered with the features we want
- **XXX_features_columns** : The list of features WITHOUT the ids


## Tower preparation

### Item tower

We have no IDs as the dataframe is ordered.

#### Item categories

In [ ]:
# Use MultiLabelBinarizer to manage efficiently the feat column
mlb = MultiLabelBinarizer()

# Transform the feat column to a list (evaluate with python)
item_categories["feat"] = item_categories["feat"].apply(eval)

item_categories = pd.DataFrame(mlb.fit_transform(item_categories["feat"]), 
                  columns=mlb.classes_,
                  index=item_categories["video_id"])


item_categories.reset_index(drop=True, inplace=True)
item_categories[item_categories.columns] = item_categories[item_categories.columns].astype("int16")

item_categories.head(3)


#### Item daily features

We take the oldest data point for a given video_id.

Depending on the complexity, you can choose the number of features.

In [ ]:
TEXT_FEATURES = ["video_type", "upload_type"] # "visible_status"
INT_FEATURES = ['video_duration','video_width', 'video_height', 'music_id', 'video_tag_id','show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
                'play_duration', 'complete_play_cnt', 'complete_play_user_num', 'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
                'long_time_play_user_num', 'short_time_play_cnt', 'short_time_play_user_num', 'play_progress']
#  ['comment_stay_duration',
       # 'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       # 'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       # 'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       # 'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       # 'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       # 'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       # 'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       # 'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       # 'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       # 'cancel_collect_user_num']

In [ ]:
# Keep the latest date for each video_id
item_daily_features = item_daily_features.loc[item_daily_features.groupby("video_id")["date"].idxmax()].reset_index(drop=True)

# One-hot str features
text_daily_features = item_daily_features[TEXT_FEATURES]
onehotter = OneHotEncoder(handle_unknown="ignore")
onehot_array = onehotter.fit_transform(text_daily_features).toarray()

# Convert to DataFrame
text_daily_features = pd.DataFrame(
    onehot_array,
    columns=onehotter.get_feature_names_out(TEXT_FEATURES),
    index=item_daily_features.index)

# Merge the one-hot encoded features back into the original DataFrame
item_daily_features = pd.concat([item_daily_features[INT_FEATURES], text_daily_features], axis=1)

In [ ]:
# No IDs, because it is the index
item_features_map = pd.concat([item_daily_features, item_categories], axis=1)

# Column names should be str
item_features_map.columns = item_features_map.columns.map(str)

# We can keep all the columns
item_features_columns = item_features_map.columns.tolist()

### User tower

In [ ]:
user_features_columns = [
    "is_lowactive_period","is_live_streamer", "is_video_author",
    "onehot_feat0", "onehot_feat1", "onehot_feat2", "onehot_feat3",
    "onehot_feat4", "onehot_feat5", "onehot_feat6", "onehot_feat7",
    "onehot_feat8", "onehot_feat9", "onehot_feat10", "onehot_feat11", 
    "onehot_feat12", "onehot_feat13", "onehot_feat14", "onehot_feat15",
    "onehot_feat16", "onehot_feat17"
]
user_features_map = user_features[user_features_columns].copy()

user_features_map[user_features_map.columns] = user_features_map[user_features_map.columns].astype("int16")

In [ ]:
# Index is the associated IDs for quick creation
display(user_features_map.head(3))
display(item_features_map.head(3))

## Dataset preparation

For each interaction in the dataset, we will create a row of user_feature and item_feature.

For example:
```python
interaction[0] = {
    'user_id': 1,
    'video_id': 2,
    'watch_ratio': 0.9
}
```
First row of user_features_vector has user1's features, and item_vector_features has item2's features.

There will be repetitions of features, but this enforces interactions without stating the IDs.

In [ ]:
# Take the IDs, populate the vectors
interaction_vector = big_matrix.iloc[:2_000_000].copy()

# Used for later
(user_ids, video_ids) = (interaction_vector["user_id"], interaction_vector["video_id"])

# Only keep the target
interaction_vector = interaction_vector[["watch_ratio"]].values.round(2)

In [ ]:
# Create the vector with features, drop IDs as not needed by the model
user_features_vector = user_features_map.iloc[user_ids].values
item_features_vector = item_features_map.iloc[video_ids].values

In [ ]:
def sanitize_check(array: np.ndarray) -> None:
    """
    Check if the array has nan/inf values.
    Sends an error message if it does.
    """
    if np.isnan(array).any() or np.isinf(array).any():
        raise ValueError(f"The array with these values {array} has nan/inf values.")
    else:
        print("array is good")

sanitize_check(user_features_vector)
sanitize_check(item_features_vector)
sanitize_check(interaction_vector)

The vectors do not have the ids of the user and item. But they are kept in memory for later use.

In [ ]:
# No IDs here
display(user_features_vector[:2])
display(item_features_vector[1])

# The target
display(interaction_vector[:2])

## Data splitting

By splitting all the data, we keep the IDs for each user and item to validate the model and test recommendations.

In [ ]:
# We split all the same way !

user_features_train, user_features_test, \
item_features_train, item_features_test, \
y_train, y_test, \
user_ids_train, user_ids_test, \
video_ids_train, video_ids_test = train_test_split(
    user_features_vector,
    item_features_vector,
    interaction_vector,
    user_ids,
    video_ids,
    test_size=0.2,
    random_state=42
)


In [ ]:
user_features_test.shape, item_features_test.shape, y_test.shape

## Scale features

Features are scaled AFTER the train-test split, else we will have data leakage.

Once we have our feature vectors associated to each interaction, we will scale the features and target.

This will preserve the shape of the data, and change the values to a range between 0 and 1.

In [ ]:
scalerItem = StandardScaler()
item_features_train = scalerItem.fit_transform(item_features_train)
item_features_test = scalerItem.transform(item_features_test)

In [ ]:
scalerUser = StandardScaler()
user_features_train = scalerUser.fit_transform(user_features_train)
user_features_test = scalerUser.transform(user_features_test)

In [ ]:
scalerTarget = StandardScaler()
y_train = scalerTarget.fit_transform(y_train)
y_test = scalerTarget.transform(y_test)

In [ ]:
item_features_train.shape, item_features_test.shape, user_features_train.shape, user_features_test.shape, y_train.shape, y_test.shape

## Dimension reduction

We will use PCA to reduce the dimensionality of the data.

In [ ]:
# Item features
pca_item = PCA(n_components=0.95)
item_features_train = pca_item.fit_transform(item_features_train)
item_features_test = pca_item.transform(item_features_test)

In [ ]:
# User features
pca_user = PCA(n_components=0.95)
user_features_train = pca_user.fit_transform(user_features_train)
user_features_test = pca_user.transform(user_features_test)


In [ ]:
item_features_train.shape, item_features_test.shape, user_features_train.shape, user_features_test.shape

# Step 3: Model architecture

The model is cut into 4 parts:
- Data preparation and tuning
- Model training
- Model evaluation
- Model saving


## Model 1: Basic Two-Tower Model

### Model creation

In [ ]:
# Constants
user_dim = user_features_train.shape[1]
item_dim = item_features_train.shape[1]

In [ ]:
# Item tower

input_item = Input(shape=(item_dim,), name="item_input")

# Process features
item_NN = Sequential(
    [
        Dense(128, activation="relu", name="item_x"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(32, activation="relu", name="item_embedding"),
    ], name="item_NN"
)

item_embedding = item_NN(input_item)

In [ ]:
# User tower

input_user = Input(shape=(user_dim,), name="user_input")

# Process features
user_NN = Sequential(
    [
        Dense(128, activation="relu", name="user_x"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(32, activation="relu", name="user_embedding"),
    ], name="user_NN"
)
user_embedding = user_NN(input_user)

In [ ]:
combined = Concatenate()([item_embedding, user_embedding])
x = Dense(64, activation="relu")(combined)
x = Dropout(0.2)(x)
x = Dense(32, activation="relu")(x)
output = Dense(1)(x)


model = Model(inputs=[input_user, input_item], outputs=output, name="Two_Tower_Model")

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"],
)

model.summary()

In [ ]:
keras.utils.plot_model(model, dpi=50, show_shapes=True, show_layer_activations=True)

### Training

In [ ]:
history = model.fit(
    x=[user_features_train, item_features_train],
    y=y_train,
    validation_data=([user_features_test, item_features_test], y_test),
    epochs=20,
    batch_size=512,
    callbacks=[keras.callbacks.EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)]
    )


### Metrics evaluation

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=history.history["val_loss"],
    mode="lines",
    name="Validation Loss"
))

fig.add_trace(go.Scatter(
    y=history.history["loss"],
    mode="lines",
    name="Train Loss"
))

fig.update_layout(
    title="Validation loss per epoch",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    legend_title="Dataset",
    xaxis=dict(tickmode="linear"),
)

In [ ]:
# Evaluate on the test set
test_loss = model.evaluate([user_features_test, item_features_test], y_test)

# Get the mean of y_train in the original scale
mean_train = scalerTarget.inverse_transform(y_train).mean()
baseline_pred = np.full_like(y_test, fill_value=mean_train)
baseline_mae = mean_squared_error(baseline_pred, y_test)

print(f"Mean value (train): {mean_train:.4f}")
print(f"Test MAE (baseline): {baseline_mae:.4f}")
print(f"Test loss (model): {test_loss[0]:.4f}")


### Saving

In [ ]:
model.save(TRAINED_MODEL)

# Step 4: Recommendation

- Predict which videos are likely to be enjoyed by each user in the test set
- Generate a top-N ranked list of recommendations for each user

### Loading model

In [ ]:
model : Model = keras.saving.load_model(TRAINED_MODEL)

### Recommendation

In [ ]:
# Reshape the inputs to add a batch dimension
test_predictions = model.predict([user_features_test, item_features_test])

In [ ]:
y_pred = scalerTarget.inverse_transform(test_predictions)
y_pred

# Step 5: Evaluation

- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG)
- Evaluate performance and provide interpretations

In [ ]:
# Predict

# Create result DataFrame
results_df = pd.DataFrame({
    "user_id": user_ids_test,
    "video_id": video_ids_test,
    "true_watch_ratio":scalerTarget.inverse_transform(y_test).flatten(),
    "predicted_watch_ratio": y_pred.flatten()
})

results_df["distance"] = abs(results_df["predicted_watch_ratio"] - results_df["true_watch_ratio"])

In [ ]:
results_df

In [ ]:

print(f"MAE: {abs(results_df["predicted_watch_ratio"] - results_df["true_watch_ratio"]).mean():.4f}")
print(f"Relative to sending the mean value: {abs([mean_train] * len(results_df) - results_df["true_watch_ratio"]).mean():.4f}")
